# CoffeeGuard AI — dataset audit

This notebook inspects the committed, group-aware split metadata and the completed EDA reports. It is intentionally runnable from a fresh clone without downloading the original Kaggle images.

**Questions answered**
- How many examples are in each split and class?
- Are groups disjoint across train/validation/test?
- What data-cleaning decisions shaped the final 2,520-image dataset?
- Which quality and resolution differences should be treated as modelling risks?

For a full raw-image re-creation, run `uv run coffeeguard data download`, then `uv run coffeeguard data prepare` before using the project CLI.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Project root: {ROOT}")


In [ ]:
SPLITS = ROOT / "data" / "splits"
REPORTS = ROOT / "artifacts" / "reports"
FIGURES = ROOT / "artifacts" / "figures" / "eda"

splits = {name: pd.read_csv(SPLITS / f"{name}.csv") for name in ["train", "val", "test"]}
summary = pd.DataFrame({
    name: df["label"].value_counts().reindex(["Healthy", "Cercospora", "Leaf Rust", "Phoma"], fill_value=0)
    for name, df in splits.items()
}).T
summary["total"] = summary.sum(axis=1)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
summary.drop(columns="total").plot.bar(ax=ax, color=["#4c956c", "#f4a261", "#d97706", "#7f5539"])
ax.set_title("Class distribution by split")
ax.set_xlabel("Split")
ax.set_ylabel("Images")
ax.legend(title="Label", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Group leakage check

The split procedure groups near-duplicate or related images before splitting. No group should appear in more than one split.


In [ ]:
group_sets = {name: set(df["group_id"]) for name, df in splits.items()}
leakage = {
    f"{a}-{b}": sorted(group_sets[a] & group_sets[b])
    for i, a in enumerate(group_sets)
    for b in list(group_sets)[i + 1:]
}
leakage_counts = {pair: len(groups) for pair, groups in leakage.items()}
print(leakage_counts)
assert not any(leakage_counts.values()), "Group leakage detected"
print("PASS: all split group IDs are disjoint.")


## Cleaning and dataset fingerprint

These values come from the committed `eda_summary.json`, so the notebook remains useful even when the large raw dataset is not present locally.


In [ ]:
eda = json.loads((REPORTS / "eda_summary.json").read_text())
headline = pd.Series({
    "scanned files": eda["files"],
    "valid files": eda["valid"],
    "final images": sum(len(df) for df in splits.values()),
    "excluded by filename pattern": eda["status_counts"]["excluded_pattern"],
    "exact duplicates removed": eda["status_counts"]["duplicate"],
    "conflicts removed": eda["status_counts"]["conflict"],
    "near-duplicate groups": eda["dedup"]["groups"],
    "imbalance ratio": eda["imbalance_ratio"],
    "dataset fingerprint": eda["fingerprint"],
}, name="value")
headline


In [ ]:
quality = eda["quality_by_class"]
quality_rows = []
for metric, per_class in quality.items():
    for label, stats in per_class.items():
        quality_rows.append({"metric": metric, "label": label, "mean": stats["mean"], "median": stats["median"]})
quality_df = pd.DataFrame(quality_rows)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["brightness", "contrast", "green_frac"]):
    data = quality_df[quality_df.metric == metric]
    sns.barplot(data=data, x="label", y="mean", ax=ax, palette="Set2", hue="label", legend=False)
    ax.set_title(f"Mean {metric}")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


In [ ]:
# Reuse the polished artifact figure when available.
eda_figure = FIGURES / "quality_by_class.png"
if eda_figure.exists():
    from IPython.display import display, Image
    display(Image(filename=str(eda_figure)))
else:
    print("EDA figure not found; use `coffeeguard data report` to regenerate it.")


### Takeaways

- The final dataset is **2,520 images** after exclusion, de-duplication, conflict removal, and grouping.
- The largest class is **Leaf Rust** and the smallest is **Phoma**; the imbalance ratio is below 2:1.
- Quality features differ substantially by class, especially sharpness and green-pixel fraction. This is a dataset-shift risk, not evidence that these features are causal disease signals.
- The test set is group-disjoint from train and validation, which makes the reported evaluation more credible than a random image-level split.
